In [ ]:
import pandas as pd
import csv
from sklearn.metrics import classification_report

In [ ]:
df_coder_dutch = pd.read_csv('cleaned_qualtrics_output_uk_train_coder_dutch.csv', encoding='utf-8-sig', sep=';', quoting=csv.QUOTE_NONNUMERIC)
df_coder_dutch['article_id'] = df_coder_dutch['article_id'].astype(int)
df_coder_dutch.article_id.nunique()
article_ids = df_coder_dutch['article_id'].tolist()

In [ ]:
# ANNOTATED DATA FOR PROMPT TESTING
df = pd.read_csv('cleaned_qualtrics_output_uk_train.csv', encoding='utf-8-sig', sep=';', quoting=csv.QUOTE_NONNUMERIC)
print(df.shape)
df['article_id'] = df['article_id'].astype(int)

df = df[df['article_id'].isin(article_ids)]
print(df.shape)

In [ ]:
df.coder.value_counts()

In [ ]:
df_coder_dutch.coder.value_counts()

In [ ]:
# change 2 to 0
df['main_topic'] = df['main_topic'].astype(int)
df['main_topic'] = df['main_topic'].replace(2, 0)
df['relevance'] = df['relevance'].astype(int, errors='ignore')
df['relevance'] = df['relevance'].replace(2, 0)

In [ ]:
df_coder_dutch['main_topic'] = df_coder_dutch['main_topic'].astype(int)
df_coder_dutch['main_topic'] = df_coder_dutch['main_topic'].replace(2, 0)
df_coder_dutch['relevance'] = df_coder_dutch['relevance'].astype(int, errors='ignore')
df_coder_dutch['relevance'] = df_coder_dutch['relevance'].replace(2, 0)

In [ ]:
# merge on article_id 
check_b = df_coder_dutch[['article_id', 'coder', 'main_topic', 'relevance']].drop_duplicates().sort_values(by=['article_id']).reset_index(drop=True)
check_e = df[['article_id', 'coder', 'main_topic', 'relevance']].drop_duplicates().sort_values(by=['article_id']).reset_index(drop=True)

check = pd.merge(check_b, check_e, on=['article_id', 'coder', 'main_topic', 'relevance'], how='outer', indicator=True)
check['_merge'].value_counts()
print(check.shape)
check.head()

pd.crosstab(check_b['main_topic'], check_e['main_topic'])

In [ ]:
# pivot check to have coder_1 and coder_2 columns for main_topic and relevance values
df_check_combined = check.pivot_table(index='article_id', columns='coder', values=['main_topic', 'relevance']).reset_index()
df_check_combined.columns = ['article_id', 'main_topic_coder_1', 'main_topic_coder_2', 'relevance_coder_1', 'relevance_coder_2']
df_check_combined.head()

# crosstab main_topic_coder_1 and main_topic_coder_2
pd.crosstab(df_check_combined['main_topic_coder_1'], df_check_combined['main_topic_coder_2'])

In [ ]:
# save this df as spss
import pyreadstat
pyreadstat.write_sav(df_check_combined, 'icr_inflation_maintopic.sav')

In [ ]:
df_check_combined[df_check_combined['main_topic_coder_1'] != df_check_combined['main_topic_coder_2']]

In [ ]:
# df_check_combined[df_check_combined['article_id'].isin([18033065, 18059348, 16000200, 16005009, 16005035, 16005258])]
# # articles not coded by LLM

In [ ]:
# df_check_combined[df_check_combined['article_id'].isin([18074222, 18098210, 18102656, 16002089, 16002851, 16003508,
#        16005090, 16005277, 17003554, 17026033, 17003783, 17003890,
#        17023779, 15015426, 15020057, 15020625, 15020769])]

# # articles coded by LLM 

In [ ]:
# combine both dataframes
df_combined = pd.concat([df, df_coder_dutch], ignore_index=True)
df_combined = df_combined.sort_values(by=['article_id', 'coder']).reset_index(drop=True)
df_combined.head()

In [ ]:
df_combined.coder.value_counts()
# make coder int
df_combined['coder'] = df_combined['coder'].astype(int)

In [ ]:
# classification report for main_topic
y_true_main = df_combined[df_combined['coder'] == 1]['main_topic']
y_pred_main = df_combined[df_combined['coder'] == 2]['main_topic']

print("Classification Report for Main Topic:")
print(classification_report(y_true_main, y_pred_main, digits=2))

In [ ]:
# calculate krippendorff's alpha for main_topic
import krippendorff
matrix = df_combined.pivot(index="coder", columns="article_id", values="main_topic")
reliability_data = matrix.to_numpy()
krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

In [ ]:
# limit to df where coder 1 and coder 2 both have coded main_topic as 1
df_relevant_1 = df_combined[(df_combined['coder'] == 1) & (df_combined['main_topic'] == 1)]
df_relevant_2 = df_combined[(df_combined['coder'] == 2) & (df_combined['main_topic'] == 1)]
# get the article ids where both coders have main_topic as 1
relevant_article_ids_1 = set(df_relevant_1['article_id'].tolist())
relevant_article_ids_2 = set(df_relevant_2['article_id'].tolist())
relevant_article_ids = relevant_article_ids_1.intersection(relevant_article_ids_2)
df_relevant = pd.concat([df_relevant_1, df_relevant_2], ignore_index=True)
df_relevant = df_relevant[df_relevant['article_id'].isin(relevant_article_ids)]
df_relevant = df_relevant.sort_values(by=['article_id', 'coder']).reset_index(drop=True)

matrix = df_relevant.pivot(index="coder", columns="article_id", values="relevance")
reliability_data = matrix.to_numpy()
krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')

In [ ]:
# classification report for relevance
y_true_main = df_relevant[df_relevant['coder'] == 1]['relevance']
y_pred_main = df_relevant[df_relevant['coder'] == 2]['relevance']

print("Classification Report for Relevance:")
print(classification_report(y_true_main, y_pred_main, digits=2))